# Lake Erie farmland, seen by NISAR radar

Radar pictures and charts of farmland along the Maumee River in Ohio, which drains into Lake Erie, made from **NASA-ISRO NISAR** L-band radar data (June to November 2026).

**What you get** (in the `outputs` folder, downloaded as one zip at the end):

| File | What it is |
|---|---|
| `viewer.html` | Clickable page: date slider, Plants / Water views, charts. Opens in any browser, no internet needed |
| `images/plants_<date>.png` | Plant cover for each date (greener = more plants) |
| `images/water_<date>.png` | Open water for each date (blue) |
| `chart_plants.png`, `chart_water.png` | Charts for slides and the demo video |
| `compare_before_after.png` | Peak season next to the latest date |
| `timeseries.csv`, `timeseries.json` | The numbers behind the charts |

**How to run:** *Runtime → Run all*. When asked, type your NASA Earthdata username and password (free account at https://urs.earthdata.nasa.gov). The radar files are several GB each, but only the small box below is streamed, so a full run usually takes 15-30 minutes, depending on NASA's servers.

In [ ]:
%pip install -q earthaccess h5py pyproj rasterio

## 1. Settings

The default box is about 30 x 30 km of farmland around Napoleon, Ohio, inside the Maumee River basin. To look somewhere else, change `BBOX` (west, south, east, north in degrees) and `AREA_NAME`.

In [ ]:
TITLE = "Lake Erie farmland, seen by radar"
AREA_NAME = "Maumee River farmland near Napoleon, Ohio"
BBOX = (-84.25, 41.25, -83.90, 41.52)      # west, south, east, north
START, END = "2026-06-17", "2026-11-15"    # calibrated NISAR data starts on 17 June 2026
MAX_DATES = 12        # dates are spread evenly over the period
PIXEL_M = 30          # analysis pixel size in metres; averaging cuts the radar "grain"
WATER_DB = -20.0      # co-pol (HH) echo below this many dB counts as open water
USE_ELEVATION = True  # also split the box into low / middle / high ground
TRACK_FRAME = None    # e.g. (44, "A", 120) to force one orbit track and frame; None = automatic
OUT_DIR = "outputs"

STORY = [
    ("Radar, not a camera.",
     "NISAR is a satellite built by NASA and ISRO, India's space agency. It sends microwaves to the ground "
     "and measures the echo, so it sees through clouds, day or night, and passes over the same place every 12 days."),
    ("Plants view:",
     "the {cross} echo bounces around inside leaves and stalks, so it gets stronger as crops grow. Dark green "
     "means lots of plants; pale means bare or smooth ground. Watch the fields fill in over summer and empty after harvest."),
    ("Water view:",
     "calm water acts like a mirror and bounces the radar away from the satellite, so it sends back almost no "
     "{co} echo. Pixels below the cut-off are painted blue."),
    ("Why it matters for Lake Erie:",
     "this farmland drains through the Maumee River into Lake Erie. Bare fields can lose soil and fertilizer when "
     "it rains, and that fertilizer feeds the toxic algae blooms at the western end of the lake."),
]
CREDITS = [
    "Radar: NASA-ISRO NISAR L2 GCOV (provisional release), distributed by NASA's Alaska Satellite Facility DAAC.",
    "Heights: Copernicus DEM GLO-30, produced using Copernicus WorldDEM-30 (c) DLR e.V. 2010-2014 and "
    "(c) Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA.",
    "Made for the 2026 NASA Space Apps Challenge.",
]

## 2. Helper functions

Nothing to change here. Run it and move on.

In [ ]:
import base64
import csv
import io
import json
import math
import os
import re
import warnings
from contextlib import contextmanager
from datetime import datetime

import h5py
import matplotlib
import numpy as np
from PIL import Image
from pyproj import Transformer

GRID = "/science/LSAR/GCOV/grids/frequencyA"
CO_KEYS = ("HHHH", "VVVV")
CROSS_KEYS = ("HVHV", "VHVH")
SHORT_NAMES = ("NISAR_L2_GCOV_PROVISIONAL_V1", "NISAR_L2_GCOV_BETA_V1")

DEM_URL = (
    "https://copernicus-dem-30m.s3.eu-central-1.amazonaws.com/"
    "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM/"
    "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM.tif"
)

_NAME = re.compile(
    r"NISAR_L2_(?P<kind>PR|UR|OD)_GCOV_(?P<cycle>\d{3})_(?P<track>\d{3})_"
    r"(?P<direction>[AD])_(?P<frame>\d{3})_.*?_(?P<start>\d{8}T\d{6})_"
    r"(?P<end>\d{8}T\d{6})_(?P<crid>[A-Z]\d{5})"
)
_KIND_RANK = {"PR": 0, "OD": 1, "UR": 2}


# ---------------------------------------------------------------- finding files

def parse_name(name):
    """Pull track, frame, date and release id out of a NISAR GCOV file name."""
    m = _NAME.search(name)
    if not m:
        return None
    d = m.groupdict()
    d["start"] = datetime.strptime(d["start"], "%Y%m%dT%H%M%S")
    for k in ("cycle", "track", "frame"):
        d[k] = int(d[k])
    return d


def _crid_key(crid):
    # higher number = newer processing; P (production) beats X (experimental)
    return (int(crid[1:]), crid[0] == "P")


def dedupe(items):
    """Keep one file per acquisition: standard product first, newest release."""
    items = sorted(items, key=lambda it: _crid_key(it["crid"]), reverse=True)
    items = sorted(items, key=lambda it: _KIND_RANK.get(it["kind"], 9))
    best = {}
    for it in items:
        key = (it["track"], it["direction"], it["frame"], it["start"].date())
        best.setdefault(key, it)
    return sorted(best.values(), key=lambda it: it["start"])


def search_gcov(bbox, start, end, short_names=SHORT_NAMES):
    """Ask NASA Earthdata for GCOV files over the box. Uses the first collection that has any."""
    import earthaccess

    for sn in short_names:
        found = []
        results = earthaccess.search_data(short_name=sn, bounding_box=tuple(bbox), temporal=(start, end))
        for g in results:
            for url in g.data_links():
                if not url.endswith(".h5"):
                    continue
                name = os.path.basename(url)[:-3]
                if "QA" in name.upper() or not re.search(r"_\d{3}$", name):
                    continue  # quality-report files, not the product itself
                info = parse_name(name)
                if info:
                    info.update(name=name, src=url, collection=sn)
                    found.append(info)
        if found:
            return dedupe(found)
    return []


def group_counts(items):
    counts = {}
    for it in items:
        k = (it["track"], it["direction"], it["frame"])
        counts[k] = counts.get(k, 0) + 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))


def pick_stack(items, max_dates=None, track_frame=None):
    """
    Images are only comparable when taken from the same orbit track and frame
    (same viewing angle), so keep one track/frame - the one with most dates
    unless track_frame=(track, "A"/"D", frame) is given.
    """
    counts = group_counts(items)
    if not counts:
        return []
    key = tuple(track_frame) if track_frame else next(iter(counts))
    stack = [it for it in items if (it["track"], it["direction"], it["frame"]) == key]
    stack.sort(key=lambda it: it["start"])
    if max_dates and len(stack) > max_dates:
        idx = sorted(set(np.linspace(0, len(stack) - 1, max_dates).round().astype(int)))
        stack = [stack[i] for i in idx]
    return stack


# ---------------------------------------------------------------- reading

@contextmanager
def open_h5(src, fs=None):
    """Open a local file or stream a remote one (only the parts we touch get downloaded)."""
    fobj = None
    if str(src).startswith("http"):
        if fs is None:
            import earthaccess
            fs = earthaccess.get_fsspec_https_session()
        fobj = fs.open(src, mode="rb", block_size=4 * 2**20, cache_type="blockcache")
        h5 = h5py.File(fobj, "r", rdcc_nbytes=32 * 2**20)
    else:
        h5 = h5py.File(src, "r")
    try:
        yield h5
    finally:
        h5.close()
        if fobj is not None:
            fobj.close()


def _epsg(grp):
    proj = grp["projection"]
    try:
        code = int(np.asarray(proj[()]).item())
        if code > 0:
            return code
    except (TypeError, ValueError):
        pass
    return int(np.asarray(proj.attrs["epsg_code"]).item())


def box_in_crs(bbox, epsg):
    """Box edges (lon/lat) -> min/max x and y in the file's map projection."""
    w, s, e, n = bbox
    t = np.linspace(0, 1, 25)
    lons = np.concatenate([w + (e - w) * t, np.full(t.size, e), e + (w - e) * t, np.full(t.size, w)])
    lats = np.concatenate([np.full(t.size, s), s + (n - s) * t, np.full(t.size, n), n + (s - n) * t])
    xs, ys = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True).transform(lons, lats)
    return float(np.min(xs)), float(np.max(xs)), float(np.min(ys)), float(np.max(ys))


def read_box(h5, bbox):
    """
    Read the co-pol and cross-pol layers for the box only.
    Returns north-up arrays covering the whole box; NaN where the file has no data.
    """
    g = h5[GRID]
    co_key = next((k for k in CO_KEYS if k in g), None)
    cross_key = next((k for k in CROSS_KEYS if k in g), None)
    if co_key is None or cross_key is None:
        raise ValueError("file has no co-pol + cross-pol pair in frequencyA")

    epsg = _epsg(g)
    x = g["xCoordinates"][:]
    y = g["yCoordinates"][:]
    dx = float(x[1] - x[0])
    dy = float(y[1] - y[0])  # negative when rows run north to south
    xmin, xmax, ymin, ymax = box_in_crs(bbox, epsg)

    c0 = math.ceil((xmin - x[0]) / dx)
    c1 = math.floor((xmax - x[0]) / dx) + 1
    if dy < 0:
        r0 = math.ceil((y[0] - ymax) / -dy)
        r1 = math.floor((y[0] - ymin) / -dy) + 1
    else:
        r0 = math.ceil((ymin - y[0]) / dy)
        r1 = math.floor((ymax - y[0]) / dy) + 1

    co = np.full((r1 - r0, c1 - c0), np.nan, dtype="float32")
    cross = np.full_like(co, np.nan)
    rr0, rr1 = max(r0, 0), min(r1, y.size)
    cc0, cc1 = max(c0, 0), min(c1, x.size)
    if rr1 > rr0 and cc1 > cc0:
        dst = (slice(rr0 - r0, rr1 - r0), slice(cc0 - c0, cc1 - c0))
        co[dst] = g[co_key][rr0:rr1, cc0:cc1]
        cross[dst] = g[cross_key][rr0:rr1, cc0:cc1]

    xs = x[0] + dx * np.arange(c0, c1)
    ys = y[0] + dy * np.arange(r0, r1)
    if dy > 0:
        co, cross, ys = co[::-1], cross[::-1], ys[::-1]
    for a in (co, cross):
        a[~np.isfinite(a) | (a <= 0)] = np.nan
    return dict(epsg=epsg, x=xs, y=ys, px=abs(dx), co=co, cross=cross,
                co_pol=co_key[:2], cross_pol=cross_key[:2])


def multilook(a, k):
    """Average k x k blocks. Cuts speckle (radar 'grain') and makes the arrays smaller."""
    if k <= 1:
        return a
    r, c = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a[:r, :c].reshape(r // k, k, c // k, k), axis=(1, 3)).astype("float32")


def _block_centres(v, k):
    n = (v.size // k) * k
    return v[:n].reshape(-1, k).mean(axis=1)


def _align(ref, arr, x, y):
    """Put arr (with its own x/y) onto the reference grid. Shifts are whole pixels."""
    out = np.full(ref["shape"], np.nan, dtype="float32")
    dc = int(round((x[0] - ref["x"][0]) / ref["px"]))
    dr = int(round((ref["y"][0] - y[0]) / ref["px"]))
    h, w = arr.shape
    H, W = ref["shape"]
    rs, cs = max(0, -dr), max(0, -dc)
    rd, cd = max(0, dr), max(0, dc)
    nh, nw = min(h - rs, H - rd), min(w - cs, W - cd)
    if nh > 0 and nw > 0:
        out[rd:rd + nh, cd:cd + nw] = arr[rs:rs + nh, cs:cs + nw]
    return out


def build_stack(stack, bbox, pixel_m=30, min_valid=0.6, fs=None, log=print):
    """Read every date in the stack onto one common grid."""
    ref = None
    dates, cos, crosses, names = [], [], [], []
    for it in stack:
        day = it["start"].strftime("%Y-%m-%d")
        try:
            with open_h5(it["src"], fs) as h5:
                lay = read_box(h5, bbox)
        except Exception as exc:  # one bad file shouldn't stop the run
            log(f"  {day}: could not read ({exc})")
            continue
        if ref is not None and lay["epsg"] != ref["epsg"]:
            log(f"  {day}: different map projection, skipped")
            continue

        k = max(1, int(round(pixel_m / lay["px"])))
        co, cross = multilook(lay["co"], k), multilook(lay["cross"], k)
        x, y = _block_centres(lay["x"], k), _block_centres(lay["y"], k)
        if ref is None:
            ref = dict(epsg=lay["epsg"], x=x, y=y, px=lay["px"] * k, shape=co.shape,
                       co_pol=lay["co_pol"], cross_pol=lay["cross_pol"])
        else:
            co, cross = _align(ref, co, x, y), _align(ref, cross, x, y)

        valid = float(np.isfinite(cross).mean())
        if valid < min_valid:
            log(f"  {day}: only {valid:.0%} of the box has data, skipped")
            continue
        dates.append(it["start"])
        cos.append(co)
        crosses.append(cross)
        names.append(it.get("name", os.path.basename(str(it["src"]))))
        log(f"  {day}: ok ({valid:.0%} of the box covered)")

    if not dates:
        raise RuntimeError("none of the files covered the box - try another area or track/frame")
    return dict(ref=ref, dates=dates, co=np.stack(cos), cross=np.stack(crosses), names=names)


# ---------------------------------------------------------------- elevation

def dem_tile_url(lat, lon):
    return DEM_URL.format(ns="N" if lat >= 0 else "S", lat=abs(lat),
                          ew="E" if lon >= 0 else "W", lon=abs(lon))


def dem_on_grid(bbox, ref, tile_url=dem_tile_url, log=print):
    """Copernicus 30 m heights (metres above sea level) resampled onto the radar grid."""
    import rasterio
    from affine import Affine
    from rasterio.warp import Resampling, reproject

    w, s, e, n = bbox
    px = ref["px"]
    transform = Affine(px, 0, ref["x"][0] - px / 2, 0, -px, ref["y"][0] + px / 2)
    out = np.full(ref["shape"], np.nan, dtype="float32")
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        for lat in range(math.floor(s), math.ceil(n)):
            for lon in range(math.floor(w), math.ceil(e)):
                url = tile_url(lat, lon)
                path = "/vsicurl/" + url if url.startswith("http") else url
                tmp = np.full(ref["shape"], np.nan, dtype="float32")
                try:
                    with rasterio.open(path) as src:
                        reproject(rasterio.band(src, 1), tmp, dst_transform=transform,
                                  dst_crs=f"EPSG:{ref['epsg']}", dst_nodata=np.nan,
                                  resampling=Resampling.bilinear)
                except Exception as exc:
                    log(f"  height tile {lat},{lon} not loaded ({exc})")
                    continue
                out = np.where(np.isnan(out), tmp, out)
    return out


def elevation_zones(dem, n=3):
    """Split the box into equal-area height zones. Returns (zone index per pixel, labels)."""
    v = dem[np.isfinite(dem)]
    if v.size < 100:
        return None, None
    edges = np.quantile(v, np.linspace(0, 1, n + 1))
    names = ["low ground", "middle ground", "high ground"] if n == 3 else [f"zone {i + 1}" for i in range(n)]
    zones = np.full(dem.shape, -1, dtype=int)
    for i in range(n):
        upper = dem <= edges[i + 1] if i == n - 1 else dem < edges[i + 1]
        zones[(dem >= edges[i]) & upper] = i
    labels = [f"{names[i]} ({edges[i]:.0f}-{edges[i + 1]:.0f} m)" for i in range(n)]
    return zones, labels


# ---------------------------------------------------------------- numbers

def db(a):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(a)


def mean_db(a, mask=None):
    """Average in power, then convert to dB (averaging dB values directly is biased)."""
    v = a if mask is None else a[mask]
    v = v[np.isfinite(v)]
    return round(float(db(v.mean())), 2) if v.size else None


def common_mask(res, min_share=0.3):
    """Pixels that have data on every date - so every date is measured over the same ground."""
    m = np.all(np.isfinite(res["co"]) & np.isfinite(res["cross"]), axis=0)
    return m if m.mean() >= min_share else None


def summarize(res, water_db=-20.0, zones=None, labels=None):
    common = common_mask(res)
    rows = []
    for i, d in enumerate(res["dates"]):
        co, cross = res["co"][i], res["cross"][i]
        valid = common if common is not None else np.isfinite(co) & np.isfinite(cross)
        water = valid & (db(co) < water_db)
        row = {
            "date": d.strftime("%Y-%m-%d"),
            "cross_db": mean_db(cross, valid),
            "co_db": mean_db(co, valid),
            "water_pct": round(100.0 * float(water.sum()) / max(int(valid.sum()), 1), 2),
            "covered_pct": round(100.0 * float(np.isfinite(cross).mean()), 1),
        }
        if zones is not None:
            for z in range(len(labels)):
                row[f"cross_db_zone{z + 1}"] = mean_db(cross, valid & (zones == z))
        rows.append(row)
    return rows


def write_table(rows, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "timeseries.csv"), "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        wr.writeheader()
        wr.writerows(rows)
    with open(os.path.join(out_dir, "timeseries.json"), "w") as f:
        json.dump(rows, f, indent=2)


# ---------------------------------------------------------------- pictures

def color_ranges(res):
    """One colour scale for all dates, so pictures can be compared fairly."""
    lo_x, hi_x = np.nanpercentile(db(res["cross"]), [2, 98])
    lo_c, hi_c = np.nanpercentile(db(res["co"]), [2, 98])
    return dict(cross=(float(lo_x), float(hi_x)), co=(float(lo_c), float(hi_c)))


def plants_rgba(cross, rng, cmap="YlGn"):
    lo, hi = rng
    v = np.clip((db(cross) - lo) / (hi - lo), 0, 1)
    rgba = (matplotlib.colormaps[cmap](np.nan_to_num(v)) * 255).astype("uint8")
    rgba[~np.isfinite(cross)] = 0
    return rgba


def water_rgba(co, rng, water_db):
    lo, hi = rng
    d = db(co)
    gray = (np.nan_to_num(np.clip((d - lo) / (hi - lo), 0, 1)) * 255).astype("uint8")
    rgba = np.dstack([gray, gray, gray, np.full_like(gray, 255)])
    rgba[np.isfinite(d) & (d < water_db)] = (30, 120, 255, 255)
    rgba[~np.isfinite(d)] = 0
    return rgba


def save_png(rgba, path):
    Image.fromarray(rgba, "RGBA").save(path, optimize=True)


def data_uri(rgba, max_side=800, bg=(233, 236, 239)):
    im = Image.fromarray(rgba, "RGBA")
    if max(im.size) > max_side:
        f = max_side / max(im.size)
        im = im.resize((max(1, round(im.width * f)), max(1, round(im.height * f))), Image.LANCZOS)
    base = Image.new("RGB", im.size, bg)
    base.paste(im, mask=im.split()[3])
    buf = io.BytesIO()
    base.save(buf, "JPEG", quality=80, optimize=True)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("ascii")


def save_pictures(res, out_dir, water_db, rng=None):
    """Full-size PNGs for slides plus small embedded copies for the viewer page."""
    rng = rng or color_ranges(res)
    img_dir = os.path.join(out_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    plants, water = [], []
    for i, d in enumerate(res["dates"]):
        day = d.strftime("%Y-%m-%d")
        p = plants_rgba(res["cross"][i], rng["cross"])
        wtr = water_rgba(res["co"][i], rng["co"], water_db)
        save_png(p, os.path.join(img_dir, f"plants_{day}.png"))
        save_png(wtr, os.path.join(img_dir, f"water_{day}.png"))
        plants.append(data_uri(p))
        water.append(data_uri(wtr))
    return plants, water, rng


def plot_charts(rows, labels, cross_pol, co_pol, water_db, out_dir):
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt

    dates = [datetime.strptime(r["date"], "%Y-%m-%d") for r in rows]
    figs = []

    fig, ax = plt.subplots(figsize=(8, 4.2), dpi=150)
    ax.plot(dates, [r["cross_db"] for r in rows], marker="o", lw=2.4, color="#2e7d32", label="whole area")
    zone_colors = ["#8d6e63", "#f9a825", "#1565c0"]
    for z, lab in enumerate(labels or []):
        ax.plot(dates, [r.get(f"cross_db_zone{z + 1}") for r in rows], marker=".", lw=1.4,
                ls="--", color=zone_colors[z % 3], label=lab)
    ax.set_title("Plant cover seen by radar")
    ax.set_ylabel(f"{cross_pol} brightness (dB)  - higher = more plants")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "chart_plants.png"))
    figs.append(fig)

    fig, ax = plt.subplots(figsize=(8, 3.4), dpi=150)
    ax.bar(dates, [r["water_pct"] for r in rows], width=4, color="#1e78ff")
    ax.set_title(f"Open water in the area (pixels with {co_pol} below {water_db:g} dB)")
    ax.set_ylabel("% of area")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.grid(alpha=0.3, axis="y")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "chart_water.png"))
    figs.append(fig)
    return figs


def compare_dates(res, rng, rows, out_dir):
    """Side by side: the date with the most plants and the latest date."""
    import matplotlib.pyplot as plt

    vals = [r["cross_db"] if r["cross_db"] is not None else -np.inf for r in rows]
    peak, last = int(np.argmax(vals)), len(rows) - 1
    pair = [(peak, "most plants"), (last, "latest")] if peak != last else [(0, "first"), (last, "latest")]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5.4), dpi=150)
    for ax, (i, tag) in zip(axes, pair):
        ax.imshow(plants_rgba(res["cross"][i], rng["cross"]))
        ax.set_title(f"{res['dates'][i]:%d %b %Y} ({tag})")
        ax.axis("off")
    fig.suptitle("Radar view of the same fields (greener = more plants)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "compare_before_after.png"))
    return fig


# ---------------------------------------------------------------- viewer page

def write_viewer(path, template, *, title, area, bbox, rows, labels, plants, water,
                 cross_pol, co_pol, water_db, story, credits):
    w, s, e, n = bbox
    data = {
        "title": title,
        "area": area,
        "story": [{"lead": lead, "text": txt} for lead, txt in story],
        "bbox": [w, s, e, n],
        "centre": [round((s + n) / 2, 4), round((w + e) / 2, 4)],
        "dates": [r["date"] for r in rows],
        "cross": [r["cross_db"] for r in rows],
        "water": [r["water_pct"] for r in rows],
        "zones": [{"label": lab, "values": [r.get(f"cross_db_zone{z + 1}") for r in rows]}
                  for z, lab in enumerate(labels or [])],
        "images": {"plants": plants, "water": water},
        "pols": {"co": co_pol, "cross": cross_pol},
        "waterDb": water_db,
        "credits": credits,
    }
    blob = json.dumps(data).replace("</", "<\\/")
    html = template.replace("__TITLE__", _esc(title)).replace("__DATA__", blob)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path


def _esc(s):
    return (str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;"))


VIEWER_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<style>
  :root { --ink:#1f2933; --muted:#5f6b7a; --line:#e3e7ec; --bg:#f6f7f9; --card:#fff; --green:#2e7d32; --blue:#1e78ff; }
  * { box-sizing: border-box; }
  body { margin: 0; font: 15px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: var(--ink); background: var(--bg); }
  header, main, footer { max-width: 1200px; margin: 0 auto; }
  header { padding: 22px 24px 6px; }
  h1 { margin: 0; font-size: 24px; letter-spacing: -0.01em; }
  h2 { margin: 0 0 8px; font-size: 16px; }
  h2 + svg { margin-bottom: 4px; }
  .sub { margin: 4px 0 0; color: var(--muted); }
  main { padding: 12px 24px 20px; display: grid; grid-template-columns: minmax(0, 1.15fr) minmax(0, 1fr); gap: 16px; }
  .card { background: var(--card); border: 1px solid var(--line); border-radius: 12px; padding: 16px; }
  .about { grid-column: 1 / -1; }
  .about p { margin: 0 0 10px; max-width: 900px; }
  .tabs { display: flex; gap: 6px; margin-bottom: 10px; }
  .tabs button { border: 1px solid var(--line); background: #fff; color: var(--ink); border-radius: 999px; padding: 5px 14px; font: inherit; cursor: pointer; }
  .tabs button.on { background: var(--ink); border-color: var(--ink); color: #fff; }
  .frame { background: #e9ecef; border-radius: 8px; overflow: hidden; }
  .frame img { display: block; width: 100%; height: auto; image-rendering: auto; }
  .legend { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; font-size: 13px; color: var(--muted); margin-top: 8px; min-height: 22px; }
  .ramp { height: 10px; width: 150px; border-radius: 5px; }
  .swatch { width: 14px; height: 14px; border-radius: 3px; display: inline-block; }
  .controls { display: flex; align-items: center; gap: 10px; margin-top: 10px; }
  .controls input[type=range] { flex: 1; accent-color: var(--green); }
  #play { width: 38px; height: 34px; border-radius: 8px; border: 1px solid var(--line); background: #fff; cursor: pointer; font-size: 13px; }
  #date { min-width: 108px; text-align: right; font-weight: 600; font-variant-numeric: tabular-nums; }
  svg { width: 100%; height: auto; display: block; overflow: visible; }
  .grid { stroke: var(--line); }
  .ylab { font-size: 11px; fill: var(--muted); text-anchor: end; }
  .xlab { font-size: 11px; fill: var(--muted); text-anchor: middle; }
  .cursor { stroke: #9aa5b1; stroke-dasharray: 3 3; }
  .dot, .bar { cursor: pointer; }
  .bar { fill: var(--blue); opacity: 0.45; }
  .bar.on { opacity: 1; }
  .key { display: flex; flex-wrap: wrap; gap: 4px 14px; font-size: 12px; color: var(--muted); margin: 2px 0 14px; }
  .key i { display: inline-block; width: 18px; height: 0; border-top: 2px solid; vertical-align: middle; margin-right: 5px; }
  .facts { font-size: 13px; color: var(--muted); border-top: 1px solid var(--line); padding-top: 10px; margin-top: 6px; }
  .small { font-size: 13px; }
  a { color: #1565c0; }
  footer { padding: 0 24px 28px; font-size: 12px; color: var(--muted); }
  footer p { margin: 2px 0; }
  @media (max-width: 860px) { main { grid-template-columns: 1fr; } }
</style>
</head>
<body>
<header>
  <h1 id="title"></h1>
  <p class="sub" id="sub"></p>
</header>

<main>
  <section class="card">
    <div class="tabs">
      <button type="button" data-view="plants" class="on">Plants</button>
      <button type="button" data-view="water">Water</button>
    </div>
    <div class="frame"><img id="pic" alt=""></div>
    <div class="legend" id="legend"></div>
    <div class="controls">
      <button type="button" id="play" aria-label="Play through the dates">&#9654;</button>
      <input type="range" id="slider" min="0" max="0" step="1" value="0" aria-label="Date">
      <span id="date"></span>
    </div>
  </section>

  <section class="card">
    <h2>Plant cover over time</h2>
    <svg id="chartPlants" viewBox="0 0 560 240" role="img" aria-label="Chart of radar plant signal by date"></svg>
    <div class="key" id="zoneKey"></div>
    <h2>Open water</h2>
    <svg id="chartWater" viewBox="0 0 560 150" role="img" aria-label="Chart of open water share by date"></svg>
    <div class="facts" id="facts"></div>
  </section>

  <section class="card about">
    <h2>What am I looking at?</h2>
    <div id="story"></div>
    <p class="small"><a id="map" target="_blank" rel="noopener">See this area on a map &#8599;</a></p>
  </section>
</main>

<footer id="credits"></footer>

<script type="application/json" id="data">__DATA__</script>
<script>
(function () {
  var D = JSON.parse(document.getElementById("data").textContent);
  var n = D.dates.length, idx = 0, view = "plants", timer = null;
  var NS = "http://www.w3.org/2000/svg";
  var MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"];
  var YLGN = ["#ffffe5", "#f7fcb9", "#d9f0a3", "#addd8e", "#78c679", "#41ab5d", "#238443", "#006837", "#004529"];
  var ZONE_COLORS = ["#8d6e63", "#f9a825", "#1565c0"];
  var t = D.dates.map(function (d) { return Date.parse(d + "T00:00:00Z"); });

  function $(id) { return document.getElementById(id); }
  function fmt(d) { var p = d.split("-"); return (+p[2]) + " " + MONTHS[+p[1] - 1] + " " + p[0]; }
  function shortDate(d) { var p = d.split("-"); return (+p[2]) + " " + MONTHS[+p[1] - 1]; }
  function el(tag, attrs, parent) {
    var e = document.createElementNS(NS, tag);
    for (var k in attrs) e.setAttribute(k, attrs[k]);
    if (parent) parent.appendChild(e);
    return e;
  }
  function text(tag, cls, str, attrs, parent) {
    attrs = attrs || {};
    attrs["class"] = cls;
    var e = el(tag, attrs, parent);
    e.textContent = str;
    return e;
  }
  function niceTicks(lo, hi, count) {
    var raw = (hi - lo) / count || 1;
    var step = Math.pow(10, Math.floor(Math.log(raw) / Math.LN10));
    var r = raw / step;
    step *= r >= 7.5 ? 10 : r >= 3.5 ? 5 : r >= 1.5 ? 2 : 1;
    var out = [];
    for (var v = Math.ceil(lo / step) * step; v <= hi + 1e-9; v += step) out.push(+v.toFixed(6));
    return out;
  }
  function xScale(left, right) {
    if (n < 2) return function () { return (left + right) / 2; };
    return function (i) { return left + (t[i] - t[0]) * (right - left) / (t[n - 1] - t[0]); };
  }
  function xLabels(svg, x, y) {
    var every = Math.max(1, Math.ceil(n / 6));
    for (var i = 0; i < n; i += every) text("text", "xlab", shortDate(D.dates[i]), { x: x(i), y: y }, svg);
  }

  // ---- header, story, credits
  $("title").textContent = D.title;
  $("sub").textContent = D.area + " · NISAR radar · " + fmt(D.dates[0]) + " to " + fmt(D.dates[n - 1]) +
    " · " + n + " dates";
  (D.story || []).forEach(function (p) {
    var para = document.createElement("p");
    if (p.lead) { var b = document.createElement("b"); b.textContent = p.lead + " "; para.appendChild(b); }
    para.appendChild(document.createTextNode(p.text));
    $("story").appendChild(para);
  });
  $("map").href = "https://www.openstreetmap.org/?mlat=" + D.centre[0] + "&mlon=" + D.centre[1] +
    "#map=11/" + D.centre[0] + "/" + D.centre[1];
  (D.credits || []).forEach(function (c) {
    var p = document.createElement("p");
    p.textContent = c;
    $("credits").appendChild(p);
  });

  // ---- plant chart
  var plants = (function () {
    var svg = $("chartPlants"), W = 560, H = 240, m = { l: 48, r: 14, t: 10, b: 26 };
    var series = [{ values: D.cross, color: "#2e7d32", width: 2.6 }];
    D.zones.forEach(function (z, i) {
      series.push({ values: z.values, color: ZONE_COLORS[i % 3], width: 1.5, dash: "5 4" });
    });
    var all = [];
    series.forEach(function (s) { s.values.forEach(function (v) { if (v !== null) all.push(v); }); });
    var lo = Math.min.apply(null, all), hi = Math.max.apply(null, all);
    var pad = (hi - lo) * 0.15 || 0.5;
    lo -= pad; hi += pad;
    var x = xScale(m.l + 14, W - m.r - 8);
    var y = function (v) { return H - m.b - (v - lo) * (H - m.b - m.t) / (hi - lo); };
    niceTicks(lo, hi, 4).forEach(function (v) {
      el("line", { x1: m.l, x2: W - m.r, y1: y(v), y2: y(v), "class": "grid" }, svg);
      text("text", "ylab", v + " dB", { x: m.l - 6, y: y(v) + 4 }, svg);
    });
    xLabels(svg, x, H - 6);
    var cursor = el("line", { y1: m.t, y2: H - m.b, "class": "cursor" }, svg);
    series.forEach(function (s) {
      var pts = [];
      s.values.forEach(function (v, i) { if (v !== null) pts.push(x(i).toFixed(1) + "," + y(v).toFixed(1)); });
      el("polyline", { points: pts.join(" "), fill: "none", stroke: s.color, "stroke-width": s.width,
        "stroke-dasharray": s.dash || "none", "stroke-linejoin": "round" }, svg);
    });
    var dots = D.cross.map(function (v, i) {
      if (v === null) return null;
      var c = el("circle", { cx: x(i), cy: y(v), r: 4.5, fill: "#fff", stroke: "#2e7d32", "stroke-width": 2, "class": "dot" }, svg);
      text("title", "", fmt(D.dates[i]) + ": " + v.toFixed(1) + " dB", {}, c);
      c.addEventListener("click", function () { stop(); setIdx(i); });
      return c;
    });
    return { cursor: cursor, dots: dots, x: x };
  })();

  if (D.zones.length) {
    var key = $("zoneKey");
    [{ label: "whole area", color: "#2e7d32" }].concat(D.zones.map(function (z, i) {
      return { label: z.label, color: ZONE_COLORS[i % 3] };
    })).forEach(function (k) {
      var s = document.createElement("span");
      var line = document.createElement("i");
      line.style.borderColor = k.color;
      s.appendChild(line);
      s.appendChild(document.createTextNode(k.label));
      key.appendChild(s);
    });
  }

  // ---- water chart
  var water = (function () {
    var svg = $("chartWater"), W = 560, H = 150, m = { l: 48, r: 14, t: 8, b: 26 };
    var hi = Math.max.apply(null, D.water.concat([1])) * 1.2;
    var x = xScale(m.l + 10, W - m.r - 10);
    var y = function (v) { return H - m.b - v * (H - m.b - m.t) / hi; };
    niceTicks(0, hi, 3).forEach(function (v) {
      el("line", { x1: m.l, x2: W - m.r, y1: y(v), y2: y(v), "class": "grid" }, svg);
      text("text", "ylab", v + "%", { x: m.l - 6, y: y(v) + 4 }, svg);
    });
    xLabels(svg, x, H - 6);
    var bw = Math.max(4, Math.min(22, (W - m.l - m.r) / Math.max(n, 1) * 0.6));
    var bars = D.water.map(function (v, i) {
      var b = el("rect", { x: x(i) - bw / 2, y: y(v), width: bw, height: Math.max(0, y(0) - y(v)), rx: 2, "class": "bar" }, svg);
      text("title", "", fmt(D.dates[i]) + ": " + v.toFixed(2) + "% open water", {}, b);
      b.addEventListener("click", function () { stop(); setIdx(i); });
      return b;
    });
    return { bars: bars };
  })();

  // ---- quick facts
  (function () {
    var best = -1;
    D.cross.forEach(function (v, i) { if (v !== null && (best < 0 || v > D.cross[best])) best = i; });
    var wet = 0;
    D.water.forEach(function (v, i) { if (v > D.water[wet]) wet = i; });
    var lines = [];
    if (best >= 0) {
      lines.push("Strongest plant signal: " + fmt(D.dates[best]) + " (" + D.cross[best].toFixed(1) + " dB).");
      var last = D.cross[n - 1];
      if (last !== null && best !== n - 1) {
        lines.push("Last date compared with that peak: " + (last - D.cross[best]).toFixed(1) + " dB.");
      }
    }
    lines.push("Most open water: " + fmt(D.dates[wet]) + " (" + D.water[wet].toFixed(2) + "% of the area).");
    $("facts").textContent = lines.join(" ");
  })();

  // ---- legend under the picture
  function legend() {
    var box = $("legend");
    box.innerHTML = "";
    function add(node) { box.appendChild(node); }
    function span(str) { var s = document.createElement("span"); s.textContent = str; return s; }
    if (view === "plants") {
      var ramp = document.createElement("span");
      ramp.className = "ramp";
      ramp.style.background = "linear-gradient(90deg," + YLGN.join(",") + ")";
      add(span("bare or smooth ground"));
      add(ramp);
      add(span("lots of plants (" + D.pols.cross + " echo)"));
    } else {
      var sw = document.createElement("span");
      sw.className = "swatch";
      sw.style.background = "#1e78ff";
      add(sw);
      add(span("open water (" + D.pols.co + " echo below " + D.waterDb + " dB)"));
      var g = document.createElement("span");
      g.className = "ramp";
      g.style.background = "linear-gradient(90deg,#000,#fff)";
      g.style.marginLeft = "10px";
      add(span("weak"));
      add(g);
      add(span("strong echo"));
    }
  }

  // ---- interaction
  var slider = $("slider");
  slider.max = Math.max(0, n - 1);
  slider.disabled = n < 2;
  slider.addEventListener("input", function () { setIdx(+slider.value); });
  Array.prototype.forEach.call(document.querySelectorAll("[data-view]"), function (b) {
    b.addEventListener("click", function () {
      view = b.getAttribute("data-view");
      Array.prototype.forEach.call(document.querySelectorAll("[data-view]"), function (o) {
        o.className = o === b ? "on" : "";
      });
      render();
    });
  });
  $("play").addEventListener("click", function () { if (timer) stop(); else start(); });
  function start() {
    if (n < 2) return;
    timer = setInterval(function () { setIdx((idx + 1) % n); }, 1000);
    $("play").innerHTML = "&#10074;&#10074;";
  }
  function stop() {
    if (timer) clearInterval(timer);
    timer = null;
    $("play").innerHTML = "&#9654;";
  }
  function setIdx(i) { idx = i; slider.value = i; render(); }

  function render() {
    var pic = $("pic");
    pic.src = D.images[view][idx];
    pic.alt = (view === "plants" ? "Radar plant view" : "Radar water view") + " on " + fmt(D.dates[idx]);
    $("date").textContent = fmt(D.dates[idx]);
    var cx = plants.x(idx);
    plants.cursor.setAttribute("x1", cx);
    plants.cursor.setAttribute("x2", cx);
    plants.dots.forEach(function (c, i) {
      if (!c) return;
      c.setAttribute("r", i === idx ? 7 : 4.5);
      c.setAttribute("fill", i === idx ? "#2e7d32" : "#fff");
    });
    water.bars.forEach(function (b, i) { b.setAttribute("class", i === idx ? "bar on" : "bar"); });
    legend();
  }

  render();
})();
</script>
</body>
</html>
'''


## 3. Log in to NASA Earthdata

Type your Earthdata username, press Enter, then your password and Enter.

In [ ]:
import earthaccess

auth = earthaccess.login(strategy="interactive", persist=True)
print("Logged in:", auth.authenticated)

## 4. Find the radar files

Pictures are only comparable when they're taken from the same orbit track and frame (same viewing angle), so one track/frame is used: the one with the most dates.

In [ ]:
items = search_gcov(BBOX, START, END)
print(f"Found {len(items)} NISAR GCOV files over the box.")
if not items:
    print("Nothing found: check the dates and box, or look the area up on https://search.asf.alaska.edu "
          "(Dataset: NISAR, Science Product: GCOV).")
for (track, direction, frame), n in group_counts(items).items():
    print(f"  track {track}, {'ascending' if direction == 'A' else 'descending'}, frame {frame}: {n} dates")

stack = pick_stack(items, MAX_DATES, TRACK_FRAME)
print(f"\nUsing {len(stack)} dates:")
for it in stack:
    print(" ", it["start"].strftime("%Y-%m-%d"), it["name"])

## 5. Read the box from each file

This is the slow part: roughly 1-2 minutes per date.

In [ ]:
fs = earthaccess.get_fsspec_https_session()
res = build_stack(stack, BBOX, pixel_m=PIXEL_M, fs=fs)
ref = res["ref"]
co_pol, cross_pol = ref["co_pol"], ref["cross_pol"]
print(f"\n{len(res['dates'])} dates ready: {ref['shape'][1]} x {ref['shape'][0]} pixels of {ref['px']:.0f} m, "
      f"{co_pol} and {cross_pol} layers.")

## 6. Low, middle and high ground

Heights come from the Copernicus 30 m elevation model (the same source NISAR's own processing uses), in metres above sea level. The box is split into three zones with the same area each.

In [ ]:
zones, labels = None, None
if USE_ELEVATION:
    dem = dem_on_grid(BBOX, ref)
    zones, labels = elevation_zones(dem)
    print("Height zones:", ", ".join(labels) if labels else "not available, continuing without them")

## 7. Numbers and charts

Every date is measured over the same pixels, so the numbers can be compared fairly.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
rows = summarize(res, water_db=WATER_DB, zones=zones, labels=labels)
write_table(rows, OUT_DIR)
for r in rows:
    print(f"{r['date']}  plants {r['cross_db']:7.2f} dB   open water {r['water_pct']:5.2f} %")
plot_charts(rows, labels, cross_pol, co_pol, WATER_DB, OUT_DIR);

## 8. Pictures

In [ ]:
plants, water, rng = save_pictures(res, OUT_DIR, WATER_DB)
compare_dates(res, rng, rows, OUT_DIR)
print(f"Saved {2 * len(plants)} pictures in {OUT_DIR}/images")

## 9. The clickable page

In [ ]:
story = [(lead, text.format(co=co_pol, cross=cross_pol)) for lead, text in STORY]
page = write_viewer(os.path.join(OUT_DIR, "viewer.html"), VIEWER_TEMPLATE, title=TITLE, area=AREA_NAME,
                    bbox=BBOX, rows=rows, labels=labels, plants=plants, water=water,
                    cross_pol=cross_pol, co_pol=co_pol, water_db=WATER_DB, story=story, credits=CREDITS)
print(f"Saved {page} ({os.path.getsize(page) / 2**20:.1f} MB)")

## 10. Download everything

In [ ]:
import shutil

zip_path = shutil.make_archive("lake_erie_radar_outputs", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; the zip is at", zip_path)